This notebook is for checking the monotonicity of the ```Center of Gravity``` defuziffication. It generates $2$  random samples, such that one is higher than the other. It is done by generating one sample. Then the other sample is generated until it is smaller that the first for every dimension. Then the inference is done for both samples and the ```CoG``` valeus are compared. If the second is bigger, monotonicity fails and code stops.

In [1]:
import numpy as np 
import pandas as pd
from tqdm import tqdm

from fuzzy_inference.rule_bases.Implicative_model import ImplicativeInferenceSystem


In [2]:
df = pd.read_csv(
    '../Data/River_quality_data_2023.csv',
    delimiter=';',
    encoding='windows-1252',   
    decimal=',' 
)
 

In [3]:
df_quaity_intervals = pd.read_csv(
    '../Data/Phisical_Chemical_quality_intervals.csv',
    delimiter=';',
    encoding='windows-1252',   
    decimal=',' 
)

In [4]:
def filter_qualityGrid_from_data(water_type = 1):
    filtered_intervals = df_quaity_intervals[df_quaity_intervals['Tips'] == water_type]
    numeric_data = filtered_intervals.select_dtypes(include='number').drop(columns=['Tips'])

    return pd.DataFrame(
        np.sort(numeric_data.values, axis=1), 
        index=numeric_data.index, 
        columns=numeric_data.columns
    ).values

In [5]:
lower_bounds = np.array([1.25, 0, 1, 0, 0])
upper_bounds = np.array([4.5, 0.225, 3.75, 10, 0.15])
num_samples = 1


In [6]:
type_1_qualityGrid = filter_qualityGrid_from_data(water_type=1)
type_1_system = ImplicativeInferenceSystem(type_1_qualityGrid, type_of_modifier='altm' )
Number_of_iterations = 10

for random_iters in tqdm(range(Number_of_iterations), desc="Running Monotonicity Tests"):
    while True:
        random_sample = np.random.uniform(low=lower_bounds, high=upper_bounds, size=(num_samples, 5))[0]
        random_sample_Smaller = np.random.uniform(low=lower_bounds, high=upper_bounds, size=(num_samples, 5))[0]
        is_second_lower = random_sample_Smaller <= random_sample
        is_first_lower = random_sample_Smaller >= random_sample
        
        if np.all(is_first_lower):
            random_sample, random_sample_Smaller = random_sample_Smaller, random_sample
        elif np.all(is_second_lower):
            break
        
    output_center_of_gravity, _  = type_1_system.inference(random_sample)
    output_center_of_gravity_smaller, _ = type_1_system.inference(random_sample_Smaller)
    assert output_center_of_gravity_smaller <= output_center_of_gravity, f"Monotonicity violated: {output_center_of_gravity_smaller} !<= {output_center_of_gravity} with inputs {random_sample_Smaller} and {random_sample}"


Running Monotonicity Tests: 100%|██████████| 10/10 [04:41<00:00, 28.12s/it]
